# Поиск CTCF и когезина у осьминогов
---

# Подготовка файлов и утилит

In [ ]:
#===============================================
# Сюда вставить данные об исследуемом организме
#===============================================

PROTEOME_URL = ""

# Примеры:
# - A. hians: https://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/054/771/915/GCA_054771915.1_Ahia01/GCA_054771915.1_Ahia01_protein.faa.gz
# - O. sinensis: https://ftp.ncbi.nlm.nih.gov/genomes/all/GCF/006/345/805/GCF_006345805.1_ASM634580v1/GCF_006345805.1_ASM634580v1_protein.faa.gz

In [ ]:
# Скачивание протеома -> proteome.faa
!wget {PROTEOME_URL}
!gunzip {PROTEOME_URL.split("/")[-1]}
!mv {PROTEOME_URL.split("/")[-1][:-3]} proteome.faa

In [ ]:
# Скачивание и установка утилиты datasets для скачивания последовательностей с NCBI
!curl -o datasets 'https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/v2/linux-amd64/datasets'
!chmod +x datasets

In [ ]:
# Установка Blast
!apt install ncbi-blast+

# Поиск

In [ ]:
# Скачивание протеинов CTCF и когезина для Octopus bimoculodies
CTCF_cohesin_proteins = {
    "XP_052828737.1": "CTCF",
    "XP_014770683.1": "SMC1",
    "XP_052828865.1": "SMC3",
    "XP_052823579.1": "RAD21"
}

proteins_ids = ",".join(CTCF_cohesin_proteins.keys())
!./datasets download gene accession {proteins_ids} --include protein --filename CTCF_cohesin.zip

In [ ]:
# Разархивация протеинов CTCF и когезина для Octopus bimoculodies
!unzip CTCF_cohesin.zip -d CTCF_cohesin/
!mv CTCF_cohesin/ncbi_dataset/data/protein.faa Octopus_bimoculodies_CTCF_cohesin.faa
!rm -r CTCF_cohesin
!rm CTCF_cohesin.zip

In [ ]:
# Создаем базу данных из белков (протеом) нашего осьминога для последующего использования BLAST
# -dbtype указывает на тип базы данных (база данных из протеинов)
# -in указывает на fasta-файл с белковыми последовательностями, из которых будет сделана база
# -out указывает на префикс файлов созданной базы данных
# Создадутся файлы: .pdb, .phr, .pin, .pot, .psq, .ptf, .pto
!makeblastdb  -dbtype prot  -in proteome.faa  -out proteins

In [ ]:
# Определяем на какие белки из протеома нашего осьминога похожи на искомые белки с помощью BLAST
# -query указывает на файл с искомымы белками (из Octopus bimoculodies)
# -db указывает на базу данных, по которой будет осуществляться поиск
# -evalue указывает на порогое значение evalue
# -outfmt указывает на формат вывода, возможные значения: 0-18, 6 соответствует табличному формату, разделитель - табуляция
# > перенаправление вывода в файл CTCF_cohesin_hits.txt
!blastp  -query Octopus_bimoculodies_CTCF_cohesin.faa  -db proteins  -evalue 1e-8  -outfmt 6  >  CTCF_cohesin_hits.txt

In [ ]:
import pandas as pd

# Читаем файл CTCF_cohesin_hits.txt
columns = [
    'Octopus bimoculodies protein',
    'My octopus protein',
    'identity',
    'length',
    'mismatch',
    'gapopen',
    'Octopus bimoculodies start',
    'Octopus bimoculodies end',
    'My octopus start',
    'My octopus end',
    'evalue', 'bitscore'
]

CTCF_cohesin_hits = pd.read_csv("CTCF_cohesin_hits.txt",
                        sep=r"\s+",
                        header=None,
                        names=columns)

# Добавляем столбец с продуктом
CTCF_cohesin_hits["product"] = CTCF_cohesin_hits["Octopus bimoculodies protein"].map(CTCF_cohesin_proteins)

# Для каждого Octopus bimoculodies protein получаем лучшую находку по evalue
CTCF_cohesin_best_hits = CTCF_cohesin_hits.loc[CTCF_cohesin_hits.groupby('product')['evalue'].idxmin()]
CTCF_cohesin_best_hits